# Mathematical error tracing

Colab workflow for the completed tracing pipeline. The runtime needs a GPU for extraction, probes, interventions, transition features, and boundary controls. CPU analysis reads frozen prediction artifacts.

The repository must already be present in the runtime or on Google Drive. This notebook does not clone repositories, read credentials, modify Git state, or contact remote services.

## Runtime and paths

In [ ]:
import json
import os
import shutil
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path
from time import monotonic

os.environ.setdefault("HF_HOME", "/content/huggingface")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "true")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

from google.colab import drive
drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/math-error-tracing")
PROJECT_NAME = "tracing-mathematical-error-detection-in-language-models"
candidates = [
    Path.cwd(),
    Path("/content") / PROJECT_NAME,
    DRIVE_ROOT / PROJECT_NAME,
    DRIVE_ROOT / "repository",
]
PROJECT_ROOT = next((path for path in candidates if (path / "pyproject.toml").exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Repository not found. Place the checkout in the runtime or under "
        f"{DRIVE_ROOT / PROJECT_NAME}."
    )
os.chdir(PROJECT_ROOT)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(PROJECT_ROOT)], check=True)

RUN_ROOT = DRIVE_ROOT
DATA_PATH = RUN_ROOT / "data/processbench.jsonl"
ARTIFACT_ROOT = RUN_ROOT / "artifacts/qwen2.5-math-1.5b-a100-bf16"
ANALYSIS_ROOT = RUN_ROOT / "artifacts/analysis"
COUNTERFACTUAL_PATH = RUN_ROOT / "data/counterfactual_pairs.jsonl"
for path in (DATA_PATH.parent, ARTIFACT_ROOT, ANALYSIS_ROOT):
    path.mkdir(parents=True, exist_ok=True)

print(f"Project: {PROJECT_ROOT}")
print(f"Run root: {RUN_ROOT}")
print(f"Artifacts: {ARTIFACT_ROOT}")
print(f"Analysis: {ANALYSIS_ROOT}")

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("A CUDA runtime is required for the model-backed stages.")
gpu_name = torch.cuda.get_device_name(0)
if not torch.cuda.is_bf16_supported():
    raise RuntimeError(f"BF16 is not supported by {gpu_name}.")
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision("high")
properties = torch.cuda.get_device_properties(0)
print(f"GPU: {gpu_name}")
print(f"Memory: {properties.total_memory / 2**30:.1f} GiB")

## Resolve the project configuration

Only runtime paths and GPU batch sizes are changed. The scientific settings come from `configs/project.yaml`.

In [ ]:
import yaml

base_config = yaml.safe_load((PROJECT_ROOT / "configs/project.yaml").read_text())
base_config["data"]["output_path"] = str(DATA_PATH)
base_config["extraction"]["output_dir"] = str(ARTIFACT_ROOT)
base_config["artifacts"]["analysis_dir"] = str(ANALYSIS_ROOT)
base_config["artifacts"]["counterfactual_pairs_path"] = str(COUNTERFACTUAL_PATH)
base_config["model"]["dtype"] = "bfloat16"
base_config["extraction"]["batch_size"] = 16
base_config["intervention"]["batch_size"] = 8
CONFIG_PATH = Path("/content/tracing_math_project.yaml")
CONFIG_PATH.write_text(yaml.safe_dump(base_config, sort_keys=False))
print(CONFIG_PATH.read_text())

In [ ]:
STATUS_PATH = RUN_ROOT / "run_status.json"
LOG_ROOT = RUN_ROOT / "logs"
LOG_ROOT.mkdir(parents=True, exist_ok=True)

def _now():
    return datetime.now(timezone.utc).isoformat()

def _write_status(status):
    temporary = STATUS_PATH.with_suffix(".tmp")
    temporary.write_text(json.dumps(status, indent=2))
    temporary.replace(STATUS_PATH)

def run_cli(*arguments, log_name=None):
    command = [sys.executable, "-m", "tracing_math", "--config", str(CONFIG_PATH), *arguments]
    stage = log_name or arguments[0].replace("-", "_")
    log_path = LOG_ROOT / f"{stage}.log"
    status = json.loads(STATUS_PATH.read_text()) if STATUS_PATH.exists() else {"stages": {}}
    status.setdefault("stages", {})
    started_at = _now()
    status.update({"current_stage": stage, "status": "running", "updated_at": started_at})
    status["stages"][stage] = {"status": "running", "started_at": started_at, "log": str(log_path)}
    _write_status(status)
    print("$", " ".join(command), flush=True)
    started = monotonic()
    with log_path.open("a", encoding="utf-8", buffering=1) as log_file:
        process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="", flush=True)
            log_file.write(line)
        return_code = process.wait()
    finished_at = _now()
    status_value = "complete" if return_code == 0 else "failed"
    status["stages"][stage].update({"status": status_value, "finished_at": finished_at, "elapsed_seconds": round(monotonic() - started, 1)})
    status.update({"current_stage": None, "status": status_value, "updated_at": finished_at})
    _write_status(status)
    if return_code:
        raise subprocess.CalledProcessError(return_code, command)
    return status_value

run_cli("validate-config")

## Data and activation states

Download and extract once. The commands resume from existing files and checkpoints.

In [ ]:
run_cli("download-data")
run_cli("extract-activations")
progress = ARTIFACT_ROOT / "extraction_progress.json"
if progress.exists():
    print(progress.read_text())

## Predictive probes and controls

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

run_cli("fit-probes")
PROBE_ROOT = ARTIFACT_ROOT / "probes"
directions = np.load(PROBE_ROOT / "directions.npz")
selected_layer = int(directions["selected_layer"])
layer_metrics = pd.read_csv(PROBE_ROOT / "layer_metrics.csv")
bootstrap = pd.read_csv(PROBE_ROOT / "test_group_bootstrap_summary.csv")
controls = pd.read_csv(PROBE_ROOT / "controls.csv")
transfer = pd.read_csv(PROBE_ROOT / "domain_transfer.csv")
print(f"Selected layer: {selected_layer}")
display(layer_metrics[(layer_metrics.layer == selected_layer) & layer_metrics.split.isin(["test", "test_error_traces"])])
display(bootstrap)
display(controls)
display(transfer.pivot(index="train_source", columns="test_source", values="auroc").round(3))
display(transfer.pivot(index="train_source", columns="test_source", values="process_f1").round(3))

## Verdict assay and figures

The intervention command checks the native readout before interpreting direction effects. The result is stored even when the validity gate fails.

In [ ]:
run_cli("run-interventions")
run_cli("render-figures")
INTERVENTION_ROOT = ARTIFACT_ROOT / "interventions"
verdict = json.loads((INTERVENTION_ROOT / "behavioral_verdict.json").read_text())
summary = pd.read_csv(INTERVENTION_ROOT / "summary.csv")
effects = pd.read_csv(INTERVENTION_ROOT / "effect_statistics.csv")
print("Native readout")
display(pd.Series(verdict).to_frame("value"))
print("Learned direction")
display(summary[summary.direction_type == "learned"])
display(effects[effects.status == "reported"])

## Frozen-score analysis

This stage does not load the model or activation shards. It reads saved predictions and intervention records.

In [ ]:
run_cli("analyze")
analysis_summary_path = ANALYSIS_ROOT / "summary.json"
if analysis_summary_path.exists():
    analysis_summary = json.loads(analysis_summary_path.read_text())
    print(json.dumps({
        "test_traces": analysis_summary.get("test_traces"),
        "test_rows": analysis_summary.get("test_rows"),
        "temporal_randomization": analysis_summary.get("temporal_randomization"),
        "matched_placebo": analysis_summary.get("matched_placebo"),
        "within_trace_discrimination": analysis_summary.get("within_trace_discrimination"),
        "causal_assay": analysis_summary.get("causal_assay"),
    }, indent=2))
display(pd.read_csv(ANALYSIS_ROOT / "main_comparison_table.csv"))
display(pd.read_csv(ANALYSIS_ROOT / "probe_control_paired_intervals.csv").head(20))
display(pd.read_csv(ANALYSIS_ROOT / "length_aware_threshold_results.csv"))

## Conditional hidden-state comparison

In [ ]:
run_cli("fit-conditional")
CONDITIONAL_ROOT = ANALYSIS_ROOT / "conditional_hidden_state"
conditional_metrics = pd.read_csv(CONDITIONAL_ROOT / "metrics.csv")
conditional_differences = pd.read_csv(CONDITIONAL_ROOT / "paired_differences.csv")
display(conditional_metrics[["condition", "auroc", "average_precision", "log_loss", "process_f1", "error_exact", "correct_rejection", "complete_accuracy"]].round(4))
display(conditional_differences[conditional_differences.metric.isin(["auroc", "log_loss", "process_f1"])].round(4))

## Matched transitions

In [ ]:
run_cli("fit-transition")
run_cli("transition-diagnostics")
TRANSITION_ROOT = ANALYSIS_ROOT / "transition_probe"
display(pd.read_csv(TRANSITION_ROOT / "bootstrap_summary.csv"))
display(pd.read_csv(TRANSITION_ROOT / "controls.csv"))
display(pd.read_csv(TRANSITION_ROOT / "matching_diagnostics.csv"))

## Boundary location

In [ ]:
run_cli("extract-boundary-controls")
run_cli("analyze-boundary-controls")
BOUNDARY_ROOT = ANALYSIS_ROOT / "boundary_control"
display(pd.read_csv(BOUNDARY_ROOT / "metrics.csv"))
display(pd.read_csv(BOUNDARY_ROOT / "paired_differences.csv"))

## Counterfactual annotation gate

The template is generated from selected traces. A reviewer must fill `corrected_step` and set `verified` to `true`. No patching result is reported until verified pairs exist.

In [ ]:
run_cli("prepare-counterfactuals")
if not COUNTERFACTUAL_PATH.exists():
    raise FileNotFoundError(COUNTERFACTUAL_PATH)
with COUNTERFACTUAL_PATH.open() as handle:
    rows = [json.loads(line) for line in handle if line.strip()]
verified = sum(bool(row.get("verified")) for row in rows)
print(f"Verified pairs: {verified}/{len(rows)}")
if verified:
    print("Verified pairs are available. Review the complete template before any patching run.")
else:
    print("No patching run: the template still requires human verification.")

## Artifact inventory

In [ ]:
required = [
    PROBE_ROOT / "layer_metrics.csv",
    PROBE_ROOT / "test_predictions.csv",
    PROBE_ROOT / "controls.csv",
    PROBE_ROOT / "domain_transfer.csv",
    INTERVENTION_ROOT / "behavioral_verdict.json",
    INTERVENTION_ROOT / "effect_statistics.csv",
    ANALYSIS_ROOT / "summary.json",
    ANALYSIS_ROOT / "main_comparison_table.csv",
    CONDITIONAL_ROOT / "metrics.csv",
    CONDITIONAL_ROOT / "paired_differences.csv",
    TRANSITION_ROOT / "bootstrap_summary.csv",
    TRANSITION_ROOT / "matching_diagnostics.csv",
    BOUNDARY_ROOT / "paired_differences.csv",
]
inventory = pd.DataFrame({"path": [str(path) for path in required], "exists": [path.exists() for path in required]})
display(inventory)
if not inventory.exists.all():
    raise RuntimeError("One or more completed pipeline outputs are missing.")